# 02. Road Backbone Analysis & Foundation

This notebook provides a detailed walkthrough of the **Backbone Foundation** creation process. We transform raw geospatial road data into a discrete point-based network enriched with traffic intensity, electrical capacity, and proximity to existing infrastructure.

## Objectives
1. **Discretize** LineString geometries from KMZ files into equidistant points (every 200m).
2. **Map Traffic** intensity from high-resolution road segments to our backbone points.
3. **Analyze Proximity** to existing EV charging stations and gas stations.
4. **Visualize Gaps** in the current network to identify optimal locations for new ultra-fast chargers.

In [ ]:
!git clone https://github.com/JJR9903/Iberdrola-Datathon.git

In [ ]:
cd "/content/Iberdrola-Datathon"

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
import os
import numpy as np
import toml
import sys

# Change directory to root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Add scripts directory to path to allow module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from scripts.sync_cloud import sync_standardized_data
from scripts.processing.create_backbone_foundation import (
    discretize_backbone_roads,
    map_traffic_to_points,
    assign_nearest_charging_stations,
    assign_nearest_gas_stations
)

# 1. Sync data (Ensures silver layer is present)
sync_standardized_data()

# 2. Load Config
config = toml.load('config.toml')
params = config['steps']['backbone_foundation']

print("\n🚀 Loading Standardized Data...")
gdf_roads = gpd.read_parquet(params['roads_path'])
gdf_traffic = gpd.read_parquet(params['traffic_path'])
gdf_chargers = gpd.read_parquet(params['chargers_path'])
gdf_gas = gpd.read_parquet(params['gas_stations_path'])

print(f" - Roads: {len(gdf_roads)} segments")
print(f" - Traffic: {len(gdf_traffic)} segments")
print(f" - Chargers: {len(gdf_chargers)} sites")
print(f" - Gas Stations: {len(gdf_gas)} sites")

print("\n✅ Data loaded correctly.")

In [ ]:
os.getcwd()

visual inspection of the road dataset

In [ ]:
gdf_roads.head()

visual inspection of the traffic dataset

In [ ]:
gdf_traffic.head()

visual inspection of the chargers dataset

In [ ]:
gdf_chargers.head()

visual inspection of the gas stations dataset

In [ ]:
gdf_gas.head()

## Step 1: Discretization
The raw road backbone is provided as high-level `LineString` geometries in a KMZ file. To perform precise point-based analysis (like calculating the distance to the exact nearest charger), we convert these lines into points sampled every 200 meters.

**Assumptions:**
*   A **200m interval** provides a good balance between spatial resolution and computational efficiency.
*   We preserve the original `backbone_id` to maintain traceability to the source data.

In [ ]:
params['kmz_path']

In [ ]:
params['sampling_interval_m']

In [ ]:
gdf_points = discretize_backbone_roads(
    gdf_roads,
    sampling_interval_m=params['sampling_interval_m']
)
print(f"Generated {len(gdf_points)} points along the backbone.")
gdf_points.head()

### Visualization: Base Point Network
Below we see the result of the discretization. Each red dot represents a sampling point where we will later attach traffic and infrastructure data.

In [ ]:
m0 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
gdf_plot = gdf_points.to_crs(4326).sample(min(1000, len(gdf_points)))

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='red',
        fill=True
    ).add_to(m0)
m0

## Step 2: Traffic Intensity Mapping
Traffic data is usually collected on specific road segments. We map this data to our backbone points using a spatial buffer.

**Assumptions & Logic:**
*   **Buffer Radius (50m)**: We look for road segments within 50m of each point.
*   **Neighbor Validation**: To avoid "cross-talk" from overpasses or intersecting roads, we only accept a segment if it intersects at least two consecutive points on the same backbone. This ensures the segment actually runs longitudinal to the backbone.

In [ ]:
params['traffic_shp_path']

In [ ]:
params['traffic_parquet_path']

In [ ]:
gdf_points = map_traffic_to_points(
    gdf_points,
    gdf_traffic,
    params['traffic_columns'],
    params['buffer_radius_m']
)
gdf_points[params['traffic_columns']].describe()

### Map 1: Road & Traffic
Points are colored based on their traffic intensity (`total_max`). Green indicates light traffic, yellow mid traffic, and red heavy traffic.

In [ ]:
import branca.colormap as cm

m1 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=gdf_points['total_max'].quantile(0.95))
colormap.caption = 'Traffic Intensity (Total Max)'
colormap.add_to(m1)

gdf_plot = gdf_points.to_crs(4326).sample(min(2000, len(gdf_points)))
for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=colormap(row['total_max']),
        fill=True,
        tooltip=f"Traffic: {row['total_max']}"
    ).add_to(m1)
m1

## Step 3: Infrastructure Proximity
We now calculate the distance to the nearest existing EV charging station and gas station.

**Assumptions:**
*   The nearest charger is identified using a spatial `nearest` join.
*   Distances are calculated in meters using the metric CRS (EPSG:3042).

In [ ]:
gdf_points = assign_nearest_charging_stations(
    gdf_points,
    gdf_chargers,
    params['max_distance_proximity']
)
gdf_points = assign_nearest_gas_stations(
    gdf_points,
    gdf_gas,
    params['max_distance_proximity']
)
print("Proximity analysis completed.")

### Map 2: Road & Current EV Chargers
This map highlights the areas with existing infrastructure. Points are colored by distance to the nearest charger (Green = Close, Red = Far).

In [ ]:
m2 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
dist_colormap = cm.linear.RdYlGn_09.reverse().scale(0, 50000) # 50km scale
dist_colormap.caption = 'Distance to Nearest Charger (m)'
dist_colormap.add_to(m2)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=dist_colormap(row.get('dist_charger_m', 50000)),
        fill=True
    ).add_to(m2)
m2

## Step 4: Electrical Capacity Integration
We load the electrical capacity data from substations and map it to our backbone network.

In [ ]:
capacity_path = 'data/standardized/electric_capacity.parquet'
gdf_capacity = gpd.read_parquet(capacity_path)
if gdf_capacity.crs != gdf_points.crs:
    gdf_capacity = gdf_capacity.to_crs(gdf_points.crs)

# Assign nearest substation capacity to each backbone point
gdf_points = gpd.sjoin_nearest(
    gdf_points,
    gdf_capacity[['Capacidad disponible (MW)', 'geometry']],
    how='left',
    distance_col='dist_substation_m'
)
gdf_points = gdf_points.drop_duplicates(subset=['point_id'])
print("Electrical capacity mapped to backbone.")

### Map 3: Road, EV Chargers & Electric Capacity
An integrated view showing existing chargers (Blue markers) and backbone points colored by available electrical capacity at the nearest substation.

In [ ]:
m3 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
cap_colormap = cm.linear.PuBuGn_09.scale(0, 100)
cap_colormap.caption = 'Substation Capacity (MW)'
cap_colormap.add_to(m3)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=cap_colormap(row['Capacidad disponible (MW)']),
        fill=True
    ).add_to(m3)

m3

### Map 4: Combined Infrastructure Gap Analysis
This map identifies target areas where **Traffic is High** and **Chargers are Far**, prioritizing areas with **High Electrical Capacity**.

In [ ]:
m4 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')

# Define a 'Priority' metric
# High Traffic + High Distance to Charger + High Capacity = High Priority
gdf_points['priority'] = (
    (gdf_points['total_max'] / gdf_points['total_max'].max()) *
    (gdf_points['dist_charger_m'] / gdf_points['dist_charger_m'].max()) *
    (gdf_points['Capacidad disponible (MW)'] / gdf_points['Capacidad disponible (MW)'].max())
)

prio_colormap = cm.linear.YlOrBr_09.scale(0, gdf_points['priority'].quantile(0.99))
prio_colormap.caption = 'Site Priority Index'
prio_colormap.add_to(m4)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=prio_colormap(row['priority']),
        fill=True,
        tooltip=f"Priority: {row['priority']:.2f}"
    ).add_to(m4)

m4